<a href="https://colab.research.google.com/github/mistyvisty/pcos-neurodivergence-advanced-rag-pinecone/blob/main/Advanced_RAG_Pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 PCOS × Neurodivergence — Advanced RAG with Pinecone + Hybrid Search

Author : Preeti Bhardwaj**

Upgrades the baseline [`pcos-neurodivergence-rag`](https://github.com/mistyvisty/pcos-neurodivergence-rag) (FAISS + MiniLM + LangChain) with:

1. **Pinecone** managed vector store (replaces FAISS for production-style serving)
2. **BM25** sparse retrieval running alongside dense retrieval
3. **Reciprocal Rank Fusion (RRF)** to merge dense + sparse result lists
4. **CrossEncoder reranking** (`ms-marco-MiniLM-L-6-v2`) on the fused top-10
5. **LLM query expansion** (Groq) before retrieval
6. **A/B test**: FAISS baseline vs. Pinecone hybrid pipeline on 10 PCOS questions
7. Auto-generated before/after comparison table for the README

Same source papers, same chunking strategy as the baseline — only the retrieval stack changes, so the A/B test is a fair comparison of *retrieval architecture*, not data

## Step 0 — Install dependencies


In [1]:
!pip install -q pinecone rank-bm25 sentence-transformers groq pymupdf langchain langchain-text-splitters faiss-cpu ipywidgets pandas tqdm
print("✅ Dependencies installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.2 MB/s eta 0:00:00
✅ Dependencies installed


## Step 1 — Load API keys


- `PINECONE_API_KEY`
- `GROQ_API_KEY`


In [4]:
from google.colab import userdata

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY').strip()
GROQ_API_KEY = userdata.get('GROQ_API_KEY').strip()

assert PINECONE_API_KEY, "PINECONE_API_KEY not found — add it via the key icon in the left sidebar"
assert GROQ_API_KEY, "GROQ_API_KEY not found — add it via the key icon in the left sidebar"
print("✅ Keys loaded")


✅ Keys loaded


## Step 2 — Create / connect the Pinecone index

Index spec (matches what you already created, or this cell will create it for you):
- name: `pcos-rag`
- dimension: `384` (matches `all-MiniLM-L6-v2`)
- metric: `cosine`
- serverless, AWS `us-east-1` (free tier region)


In [5]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "pcos-rag"
DIMENSION = 384

existing = [idx["name"] for idx in pc.list_indexes()]

if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"✅ Created index '{INDEX_NAME}'")
else:
    print(f"✅ Index '{INDEX_NAME}' already exists — connecting to it")

pinecone_index = pc.Index(INDEX_NAME)
print(pinecone_index.describe_index_stats())


✅ Created index 'pcos-rag'
DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


## Step 3 — Upload the papers

Same 5 PDFs as the baseline repo. Download them via the DOI links in the original README, name them exactly as below, and upload here.


In [6]:
from google.colab import files
import os

os.makedirs("papers", exist_ok=True)
print("Upload pmos.pdf, pmos1.pdf, pmos2.pdf, pmos3.pdf, pmos4.pdf")
uploaded = files.upload()

for fname in uploaded:
    os.replace(fname, f"papers/{fname}")

print("✅ Papers in papers/:", os.listdir("papers"))


Upload pmos.pdf, pmos1.pdf, pmos2.pdf, pmos3.pdf, pmos4.pdf


Saving pmos4.pdf to pmos4.pdf
Saving pmos3.pdf to pmos3.pdf
Saving pmos2.pdf to pmos2.pdf
Saving pmos1.pdf to pmos1.pdf
Saving pmos.pdf to pmos.pdf
✅ Papers in papers/: ['pmos2.pdf', 'pmos4.pdf', 'pmos.pdf', 'pmos1.pdf', 'pmos3.pdf']


## Step 4 — Extract text and chunk (identical to baseline, for a fair A/B test)


In [7]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

PAPER_META = {
    "pmos.pdf":  "Cherskov et al. 2018 — PCOS and Autism",
    "pmos1.pdf": "Redkar & Khan 2025 — PCOS and Attention",
    "pmos2.pdf": "Berni et al. 2018 — PCOS Mental Health & Neurodevelopment",
    "pmos3.pdf": "Dubey et al. 2021 — Maternal PCOS Meta-analysis",
    "pmos4.pdf": "Chen et al. 2020 — Finnish Cohort Study",
}

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

chunks = []  # list of {id, text, source, page}
chunk_id = 0

for fname, label in PAPER_META.items():
    path = f"papers/{fname}"
    if not os.path.exists(path):
        print(f"⚠️ skipping missing file: {fname}")
        continue
    doc = fitz.open(path)
    for page_num, page in enumerate(doc, start=1):
        page_text = page.get_text()
        if not page_text.strip():
            continue
        for piece in splitter.split_text(page_text):
            chunks.append({
                "id": f"chunk_{chunk_id}",
                "text": piece,
                "source": label,
                "page": page_num,
            })
            chunk_id += 1
    doc.close()

print(f"✅ Built {len(chunks)} chunks from {len(PAPER_META)} papers")


✅ Built 376 chunks from 5 papers


## Step 5 — Embed all chunks (shared embedding model: `all-MiniLM-L6-v2`)


In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True, batch_size=32)
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print(f"✅ Embedded {len(chunk_embeddings)} chunks, dim={chunk_embeddings.shape[1]}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Embedded 376 chunks, dim=384


## Step 6 — BASELINE pipeline (FAISS, top-5, no rerank, no expansion)

This is the "before" — it's the same architecture as `pcos-neurodivergence-rag`, rebuilt here from the same chunks so the A/B test is apples-to-apples.


In [9]:
import faiss
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "llama-3.3-70b-versatile"

faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
faiss_normed = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
faiss_index.add(faiss_normed)

SYSTEM_PROMPT = (
    "You are a medical research assistant. Answer ONLY using the provided context. "
    "Cite the source (author + year) and page number for every claim. "
    "If the context does not contain enough information to answer, say so explicitly — do not guess."
)

def faiss_search(query, top_k=5):
    q_emb = embed_model.encode([query]).astype("float32")
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores, idxs = faiss_index.search(q_emb, top_k)
    return [chunks[i] for i in idxs[0]]

def generate_answer(question, context_chunks):
    context = "\n\n".join(
        f"[{c['source']}, p.{c['page']}]\n{c['text']}" for c in context_chunks
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    resp = groq_client.chat.completions.create(model=GROQ_MODEL, messages=messages, temperature=0.2)
    return resp.choices[0].message.content

def baseline_rag_query(question):
    retrieved = faiss_search(question, top_k=5)
    answer = generate_answer(question, retrieved)
    return {"answer": answer, "chunks": retrieved}

print("✅ Baseline (FAISS) pipeline ready")


✅ Baseline (FAISS) pipeline ready


## Step 7 — Upsert chunks into Pinecone (dense store for the advanced pipeline)


In [10]:
BATCH_SIZE = 100

vectors = [
    {
        "id": c["id"],
        "values": chunk_embeddings[i].tolist(),
        "metadata": {"text": c["text"], "source": c["source"], "page": c["page"]},
    }
    for i, c in enumerate(chunks)
]

for i in range(0, len(vectors), BATCH_SIZE):
    pinecone_index.upsert(vectors=vectors[i:i + BATCH_SIZE])

print(f"✅ Upserted {len(vectors)} vectors into Pinecone")
print(pinecone_index.describe_index_stats())


✅ Upserted 376 vectors into Pinecone
DescribeIndexStatsResponse(dimension=384, total_vector_count=376, metric='cosine', namespaces=1)


## Step 8 — BM25 sparse retrieval (built over the same chunks)


In [11]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    top_idxs = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i]["id"], chunks[i]) for i in top_idxs]

print("✅ BM25 index ready")


✅ BM25 index ready


## Step 9 — Dense retrieval via Pinecone


In [12]:
def pinecone_dense_search(query, top_k=10):
    q_emb = embed_model.encode([query])[0].tolist()
    results = pinecone_index.query(vector=q_emb, top_k=top_k, include_metadata=True)
    return [
        (m["id"], {"text": m["metadata"]["text"], "source": m["metadata"]["source"], "page": m["metadata"]["page"]})
        for m in results["matches"]
    ]

print("✅ Pinecone dense search ready")


✅ Pinecone dense search ready


## Step 10 — Reciprocal Rank Fusion (RRF)

Merges any number of ranked result lists (dense + sparse, across all expanded query variants) into a single fused ranking. RRF score for a doc = sum over lists of `1 / (k + rank)`. We don't need calibrated similarity scores — just consistent rankings — which is what makes RRF a good fit for combining dense + sparse.


In [13]:
def rrf_fuse(result_lists, k=60, top_n=10):
    scores = {}
    doc_lookup = {}
    for results in result_lists:
        for rank, (doc_id, doc_data) in enumerate(results):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
            doc_lookup[doc_id] = doc_data
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return [(doc_id, doc_lookup[doc_id], score) for doc_id, score in fused]

print("✅ RRF fusion ready")


✅ RRF fusion ready


## Step 11 — CrossEncoder reranking

The fused top-10 are still ranked by retrieval-time signals (embedding cosine sim, BM25 score). A CrossEncoder reads the *query and passage together* and scores relevance directly — much more accurate, but too slow to run over the whole corpus, which is why it only runs on the top-10 fused candidates.


In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_with_crossencoder(query, fused_candidates, top_k=5):
    pairs = [(query, c[1]["text"]) for c in fused_candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(fused_candidates, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [
        {"id": c[0], "text": c[1]["text"], "source": c[1]["source"], "page": c[1]["page"], "rerank_score": float(s)}
        for (c, s) in ranked
    ]

print("✅ CrossEncoder reranker loaded")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ CrossEncoder reranker loaded


## Step 12 — Query expansion via Groq (before retrieval)

The LLM rewrites the question into a few alternative phrasings (synonyms, related medical terms) so retrieval isn't limited to one exact wording. Each variant gets its own dense + sparse search; all result lists feed into the same RRF fusion above.


In [15]:
def expand_query(question, n=3):
    prompt = (
        f"Rewrite this medical research question {n} different ways for a document search engine. "
        "Keep all medical terminology accurate. Vary the phrasing and synonyms, not the meaning. "
        "Return ONLY the rewritten questions, one per line, no numbering, no extra text.\n\n"
        f"Question: {question}"
    )
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=200,
    )
    lines = [l.strip("- ").strip() for l in resp.choices[0].message.content.split("\n") if l.strip()]
    return [question] + lines[:n]

print("✅ Query expansion ready")
print(expand_query("Are children of mothers with PCOS at higher risk of autism?"))


✅ Query expansion ready
['Are children of mothers with PCOS at higher risk of autism?', 'Do offspring of females with polycystic ovary syndrome have an increased likelihood of developing autistic disorder?', 'Are maternal polycystic ovarian syndrome and autism spectrum disorder in children associated with elevated risk?', 'Do children born to women diagnosed with polycystic ovary syndrome exhibit a higher incidence of autism spectrum disorders?']


## Step 13 — Full advanced pipeline (query expansion → hybrid retrieval → RRF → rerank → cited answer)

This is the "after" pipeline used in the A/B test below.


In [16]:
import time

def advanced_rag_query(question, top_k_final=5, verbose=False):
    t0 = time.time()

    expanded = expand_query(question)

    all_results = []
    for q in expanded:
        all_results.append(pinecone_dense_search(q, top_k=10))
        all_results.append(bm25_search(q, top_k=10))

    fused = rrf_fuse(all_results, k=60, top_n=10)
    reranked = rerank_with_crossencoder(question, fused, top_k=top_k_final)
    answer = generate_answer(question, reranked)

    elapsed = time.time() - t0

    if verbose:
        print(f"Expanded queries: {expanded}")
        for r in reranked:
            print(f"  [{r['rerank_score']:.3f}] {r['source']} p.{r['page']}")

    return {"answer": answer, "chunks": reranked, "expanded_queries": expanded, "time_sec": elapsed}

print("✅ Advanced pipeline ready")


✅ Advanced pipeline ready


## Step 14 — Sanity check: run one question through both pipelines


In [17]:
test_q = "Are children of mothers with PCOS at higher risk of autism?"

print("=== BASELINE (FAISS) ===")
b = baseline_rag_query(test_q)
print(b["answer"][:600], "...\n")

print("=== ADVANCED (Pinecone hybrid) ===")
a = advanced_rag_query(test_q, verbose=True)
print(a["answer"][:600], "...")


=== BASELINE (FAISS) ===
According to Chen et al. (2020, p.6), women with PCOS had 35% increased odds of having a first-born child with autism, after adjusting for comorbid maternal psychiatric diagnosis, metabolic conditions, and complications in childbirth. Additionally, Dubey et al. (2021, p.7) confirms that maternal PCOS is associated with children with ASD. Therefore, the answer is yes, children of mothers with PCOS are at higher risk of autism (Chen et al., 2020, p.6; Dubey et al., 2021, p.7). ...

=== ADVANCED (Pinecone hybrid) ===
Expanded queries: ['Are children of mothers with PCOS at higher risk of autism?', 'Do offspring of women with Polycystic Ovary Syndrome have an increased likelihood of developing autistic disorder?', 'Are maternal PCOS and autism spectrum disorder in children associated with a higher incidence rate?', 'Is there a correlation between maternal diagnosis of Polycystic Ovarian Syndrome and the risk of autism in their progeny?']
  [8.382] Chen et al. 2020 

## Step 15 — A/B test: FAISS baseline vs. Pinecone hybrid on 10 PCOS questions

Same 10 questions through both pipelines. Records latency, retrieved sources, and answers so you can eyeball groundedness and copy the table straight into the README.


In [18]:
ab_questions = [
    "Are children of mothers with PCOS at higher risk of autism?",
    "How do elevated androgens in PCOS affect brain development?",
    "What mental health disorders are most commonly associated with PCOS?",
    "Does insulin resistance in PCOS contribute to cognitive impairment?",
    "What is the prenatal sex steroid theory of autism?",
    "What did the Finnish cohort study find about maternal PCOS?",
    "Is there a link between PCOS and ADHD in offspring?",
    "What sample size did the Berni et al. study use?",
    "Does PCOS increase the risk of anxiety in affected women themselves?",
    "What neurodevelopmental outcomes were measured in the Dubey meta-analysis?",
]

import pandas as pd

rows = []
for q in ab_questions:
    t0 = time.time()
    base = baseline_rag_query(q)
    base_time = time.time() - t0

    adv = advanced_rag_query(q)

    rows.append({
        "question": q,
        "baseline_time_sec": round(base_time, 2),
        "baseline_sources": ", ".join(sorted({c["source"] for c in base["chunks"]})),
        "baseline_answer": base["answer"],
        "advanced_time_sec": round(adv["time_sec"], 2),
        "advanced_sources": ", ".join(sorted({c["source"] for c in adv["chunks"]})),
        "advanced_answer": adv["answer"],
    })
    print(f"done: {q[:60]}...")

ab_df = pd.DataFrame(rows)
ab_df.to_csv("ab_test_results.csv", index=False)
print("✅ A/B test complete — saved to ab_test_results.csv")
ab_df[["question", "baseline_time_sec", "advanced_time_sec"]]


done: Are children of mothers with PCOS at higher risk of autism?...
done: How do elevated androgens in PCOS affect brain development?...
done: What mental health disorders are most commonly associated wi...
done: Does insulin resistance in PCOS contribute to cognitive impa...
done: What is the prenatal sex steroid theory of autism?...
done: What did the Finnish cohort study find about maternal PCOS?...
done: Is there a link between PCOS and ADHD in offspring?...
done: What sample size did the Berni et al. study use?...
done: Does PCOS increase the risk of anxiety in affected women the...
done: What neurodevelopmental outcomes were measured in the Dubey ...
✅ A/B test complete — saved to ab_test_results.csv


,question,baseline_time_sec,advanced_time_sec
0,Are children of mothers with PCOS at higher ri...,1.15,3.80
1,How do elevated androgens in PCOS affect brain...,0.83,2.23
2,What mental health disorders are most commonly...,0.72,2.96
3,Does insulin resistance in PCOS contribute to ...,0.66,2.37
4,What is the prenatal sex steroid theory of aut...,0.61,3.17
5,What did the Finnish cohort study find about m...,0.41,4.02
6,Is there a link between PCOS and ADHD in offsp...,4.75,6.77
7,What sample size did the Berni et al. study use?,5.77,6.12
8,Does PCOS increase the risk of anxiety in affe...,5.79,7.19
9,What neurodevelopmental outcomes were measured...,6.73,6.00


## Step 16 — Generate the before/after comparison table for the README


In [19]:
def make_readme_table(df):
    lines = [
        "| # | Question | Baseline (FAISS) time | Advanced (Hybrid) time | Baseline sources | Advanced sources |",
        "|---|----------|----------------------|------------------------|-------------------|-------------------|",
    ]
    for i, row in df.iterrows():
        lines.append(
            f"| {i+1} | {row['question']} | {row['baseline_time_sec']}s | {row['advanced_time_sec']}s | "
            f"{row['baseline_sources']} | {row['advanced_sources']} |"
        )
    return "\n".join(lines)

table_md = make_readme_table(ab_df)
with open("comparison_table.md", "w") as f:
    f.write("## Before vs After — A/B Test Results\n\n")
    f.write(table_md)
    f.write(f"\n\n**Average latency** — baseline: {ab_df['baseline_time_sec'].mean():.2f}s | "
            f"advanced: {ab_df['advanced_time_sec'].mean():.2f}s\n")

print(table_md)
print("\n✅ Saved comparison_table.md — paste this into your repo README")


| # | Question | Baseline (FAISS) time | Advanced (Hybrid) time | Baseline sources | Advanced sources |
|---|----------|----------------------|------------------------|-------------------|-------------------|
| 1 | Are children of mothers with PCOS at higher risk of autism? | 1.15s | 3.8s | Chen et al. 2020 — Finnish Cohort Study, Dubey et al. 2021 — Maternal PCOS Meta-analysis | Chen et al. 2020 — Finnish Cohort Study, Dubey et al. 2021 — Maternal PCOS Meta-analysis |
| 2 | How do elevated androgens in PCOS affect brain development? | 0.83s | 2.23s | Berni et al. 2018 — PCOS Mental Health & Neurodevelopment, Chen et al. 2020 — Finnish Cohort Study, Cherskov et al. 2018 — PCOS and Autism, Redkar & Khan 2025 — PCOS and Attention | Berni et al. 2018 — PCOS Mental Health & Neurodevelopment, Cherskov et al. 2018 — PCOS and Autism, Dubey et al. 2021 — Maternal PCOS Meta-analysis, Redkar & Khan 2025 — PCOS and Attention |
| 3 | What mental health disorders are most commonly associated with P

## Step 17 — (Optional) Interactive Q&A widget — advanced pipeline


In [20]:
import ipywidgets as widgets
from IPython.display import display, Markdown

query_box = widgets.Text(placeholder="Ask a PCOS x neurodivergence question...", layout=widgets.Layout(width="600px"))
ask_button = widgets.Button(description="Ask (advanced)")
output_box = widgets.Output()

def on_ask_clicked(b):
    with output_box:
        output_box.clear_output()
        result = advanced_rag_query(query_box.value)
        display(Markdown(f"**Answer:**\n\n{result['answer']}"))
        display(Markdown("**Sources used:** " + ", ".join(sorted({c['source'] for c in result['chunks']}))))

ask_button.on_click(on_ask_clicked)
display(widgets.VBox([query_box, ask_button, output_box]))
